# Regresión Logística — Riesgo cardiovascular (BRFSS 2024)

Este cuaderno implementa el modelo Regresión Logística. Se emplean técnicas de validación cruzada, imputación proporcional, normalización y SMOTE.

Los datos de entrada ya vienen imputados, normalizados y balanceados con SMOTENC, generados una sola vez por `scripts/preprocesar_pliegues.py` (ver README del proyecto). Si la carpeta `data_procesada/` no existe, es necesario correr ese script antes de continuar.




In [1]:
import os
import sys
sys.path.append("..")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from scripts import pipeline as pl

DATA_DIR = "../data_procesada"
N_PLIEGUES = 5


def cargar(nombre):
    """Lee una particion ya preprocesada (generada por scripts/preprocesar_pliegues.py)."""
    tabla = pd.read_parquet(f"{DATA_DIR}/{nombre}")
    X = tabla.drop(columns=[pl.TARGET_COL])
    y = tabla[pl.TARGET_COL]
    return X, y

In [2]:
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        "No se encontro data_procesada/. Corre primero 'python scripts/preprocesar_pliegues.py' "
        "desde la raiz del proyecto (tarda unos 15 minutos, se corre una sola vez por maquina)."
    )

print("cache encontrado en", DATA_DIR)

cache encontrado en ../data_procesada


## Prueba sobre el pliegue 0

Antes de ejecutar el ciclo completo de 5 pliegues, se valida el procedimiento sobre un único pliegue, con el fin de confirmar que el modelo entrena correctamente sobre estos datos y que la función `pipeline.evaluar()` procesa sus predicciones sin errores.

In [3]:
X_train, y_train = cargar("fold_0_train.parquet")
X_test, y_test = cargar("fold_0_test.parquet")
prot_test = X_test[pl.PROTECTED_COLS]

print("train:", X_train.shape, "| balance:", y_train.value_counts().to_dict())

train: (634536, 30) | balance: {0: 317268, 1: 317268}


In [6]:
modelo = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    random_state=pl.RANDOM_STATE,
)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]